# Базовый SQL

## В этом проекте сформированы запросы к учебной базе.

In [ ]:
SELECT *
FROM company
WHERE status = 'closed'
;

In [ ]:
SELECT CAST(funding_total AS numeric)
FROM company
WHERE category_code = 'news'
AND country_code = 'USA'
ORDER BY funding_total DESC
;

In [ ]:
SELECT SUM(price_amount) AS total_transaction_amount
FROM acquisition
WHERE acquired_at BETWEEN '2011-01-01' AND '2013-12-31'
AND term_code = 'cash'
;

In [ ]:
SELECT p.first_name,
       p.last_name,
       p.network_username
FROM people AS p
WHERE p.network_username LIKE 'Silver%'
;

In [ ]:
SELECT *
FROM people
WHERE network_username LIKE '%money%'
AND last_name LIKE 'K%'
;

In [ ]:
SELECT country_code,
       SUM(funding_total) AS total_investment
FROM company
GROUP BY country_code
ORDER BY total_investment DESC
;

In [ ]:
SELECT funded_at,
       MIN(raised_amount) AS min_raised_amount,
       MAX(raised_amount) AS max_sum_raised_amount
FROM funding_round
-- WHERE raised_amount <> 0
GROUP BY funded_at
HAVING MIN(raised_amount) <> MAX(raised_amount)
AND  MIN(raised_amount) <> 0
ORDER BY funded_at, min_raised_amount
;

In [ ]:
SELECT *,
       CASE
           WHEN invested_companies >= 100 THEN 'high_activity'
           WHEN invested_companies >= 20 THEN 'middle_activity'
           ELSE 'low_activity'
       END AS activity
FROM fund
; 

In [ ]:
SELECT CASE
           WHEN invested_companies>=100 THEN 'high_activity'
           WHEN invested_companies>=20 THEN 'middle_activity'
           ELSE 'low_activity'
       END AS activity,
       ROUND(AVG(investment_rounds), 0) AS avg_activity
FROM fund
GROUP BY activity
ORDER BY avg_activity
;

In [ ]:
SELECT f.country_code,
       MIN(invested_companies) AS min_inv_comp,
       MAX(invested_companies) AS max_inv_comp,
       AVG(invested_companies) AS avg_inv_comp
FROM fund AS f
WHERE f.founded_at BETWEEN '2010-01-01'AND '2012-12-31'
GROUP BY f.country_code
HAVING MIN(invested_companies) <> 0
ORDER BY avg_inv_comp DESC, f.country_code
LIMIT 10
;

In [ ]:
SELECT p.first_name,
       p.last_name,
       e.instituition
FROM people AS p
LEFT JOIN education AS e ON p.id = e.person_id
ORDER BY e.instituition
;

In [ ]:
SELECT t.name,
       t.number_of_instituition
FROM (SELECT c.id,
             c.name,
             COUNT(DISTINCT instituition) AS number_of_instituition
      FROM education AS e
      RIGHT JOIN people AS p ON e.person_id = p.id 
      RIGHT JOIN company AS c ON p.company_id = c.id
      GROUP BY c.id, c.name
      ORDER BY number_of_instituition DESC 
      LIMIT 5) AS t
;

In [ ]:
SELECT DISTINCT c.name
FROM company AS c
WHERE c.id IN (SELECT company_id
               FROM funding_round
               WHERE is_first_round = 1
               AND is_last_round = 1)
AND c.status = 'closed'
ORDER BY c.name
;

In [ ]:
SELECT p.id 
FROM people AS p 
JOIN company AS c ON p.company_id = c.id
WHERE company_id IN (SELECT company_id
                     FROM funding_round
                     WHERE is_first_round = 1
                     AND is_last_round = 1)
AND c.status = 'closed'
ORDER BY p.id
;

In [ ]:
SELECT DISTINCT p.id,
       e.instituition AS educational_establishments
FROM people AS p 
JOIN education AS e ON p.id = e.person_id    
JOIN company AS c ON p.company_id = c.id
WHERE company_id IN (SELECT company_id
                     FROM funding_round
                     WHERE is_first_round = 1
                     AND is_last_round = 1)
AND c.status = 'closed'
ORDER BY p.id, educational_establishments
;

In [ ]:
SELECT DISTINCT p.id,
       COUNT(e.instituition) AS number_of_instituition
FROM people AS p 
JOIN education AS e ON p.id = e.person_id    
JOIN company AS c ON p.company_id = c.id
WHERE company_id IN (SELECT company_id
                     FROM funding_round
                     WHERE is_first_round = 1
                     AND is_last_round = 1)
AND c.status = 'closed'
GROUP BY p.id
ORDER BY p.id
;

In [ ]:
SELECT AVG(number_of_instituition)
FROM (SELECT DISTINCT p.id,
      COUNT(e.instituition) AS number_of_instituition
      FROM people AS p 
      JOIN education AS e ON p.id = e.person_id    
      JOIN company AS c ON p.company_id = c.id
      WHERE company_id IN (SELECT company_id
                           FROM funding_round
                           WHERE is_first_round = 1
                           AND is_last_round = 1)
      AND c.status = 'closed'
      GROUP BY p.id
      ORDER BY p.id
    ) AS n
;

In [ ]:
SELECT AVG(number_of_instituition)
FROM (SELECT DISTINCT p.id,
      COUNT(e.instituition) AS number_of_instituition
      FROM people AS p 
      JOIN education AS e ON p.id = e.person_id    
      JOIN company AS c ON p.company_id = c.id
      WHERE c.name = 'Socialnet'
      GROUP BY p.id
      ORDER BY p.id
    ) AS n
;

In [ ]:
WITH c AS (
    SELECT DISTINCT c.id,
                    c.name AS name_of_company
    FROM company AS c
    WHERE milestones > 6
            ),
     fr AS (
     SELECT fr.id,
            fr.company_id,
            fr.raised_amount AS amount
     FROM funding_round AS fr
     WHERE funded_at BETWEEN '2012-01-01'
                         AND '2013-12-31'
            ),
     f AS (
     SELECT f.id,
            f.name AS name_of_fund
     FROM fund AS f
            ),
     i AS (
     SELECT i.id,
            i.funding_round_id,
            i.fund_id
     FROM investment AS i
            )
SELECT f.name_of_fund,
       c.name_of_company,
       fr.amount
FROM c
JOIN fr ON c.id = fr.company_id
JOIN i ON fr.id = i.funding_round_id
JOIN f ON i.fund_id = f.id
ORDER BY f.name_of_fund,
         c.name_of_company,
         fr.amount
;

In [ ]:
SELECT c.name AS acquiring_company,
       a.price_amount,
       b.name AS acquired_company,
       b.funding_total,
       ROUND(a.price_amount/b.funding_total, 0) AS share_of_amount
FROM acquisition AS a
JOIN company AS c ON a.acquiring_company_id = c.id
JOIN company AS b ON a.acquired_company_id = b.id
WHERE b.funding_total <> 0
AND price_amount <> 0
ORDER BY price_amount DESC, acquired_company
LIMIT 10
;

In [ ]:
SELECT c.name AS social_companies,
       EXTRACT(MONTH FROM fr.funded_at) AS month_of_funded       
FROM company AS c
JOIN funding_round AS fr ON c.id = fr.company_id
WHERE category_code = 'social'
AND fr.funded_at BETWEEN '2010-01-01' AND '2013-12-31'
AND raised_amount <> 0
ORDER BY social_companies
;

In [ ]:
WITH f AS (
    SELECT EXTRACT(MONTH FROM funded_at) AS month_of_funded,
           COUNT(DISTINCT f.id) AS number_of_funds
    FROM funding_round AS fr
    JOIN investment AS i ON fr.id = i.funding_round_id
    JOIN fund AS f ON i.fund_id = f.id
    WHERE fr.funded_at BETWEEN '2010-01-01' AND '2013-12-31'
    AND f.country_code = 'USA'
    GROUP BY EXTRACT(MONTH FROM funded_at)
            ),
      a AS (
    SELECT EXTRACT(MONTH FROM acquired_at) AS month_of_acquired,
           COUNT(acquired_company_id) AS number_of_acquired_companies,
           SUM(price_amount) AS sum_of_investment
    FROM acquisition AS a
    WHERE acquired_at BETWEEN '2010-01-01' AND '2013-12-31'
    GROUP BY EXTRACT(MONTH FROM acquired_at)
    ORDER BY EXTRACT(MONTH FROM acquired_at)
           )

SELECT a.month_of_acquired,
       number_of_funds,
       number_of_acquired_companies,
       sum_of_investment
FROM f 
JOIN a ON f.month_of_funded = a.month_of_acquired
;

In [ ]:
WITH
     inv_2011 AS (
                  SELECT AVG(funding_total) AS avg_inv_2011,
                         country_code
                  FROM company AS c
                  WHERE EXTRACT(year FROM founded_at) = 2011
                  GROUP BY country_code
                  ),
     inv_2012 AS (
                  SELECT AVG(funding_total) AS avg_inv_2012,
                         country_code
                  FROM company AS c
                  WHERE EXTRACT(year FROM founded_at) = 2012
                  GROUP BY country_code
                  ),
     inv_2013 AS (
                  SELECT AVG(funding_total) AS avg_inv_2013,
                         country_code
                  FROM company AS c
                  WHERE EXTRACT(year FROM founded_at) = 2013
                  GROUP BY country_code
                  )
SELECT inv_2011.country_code,
       avg_inv_2011,
       avg_inv_2012,
       avg_inv_2013
FROM inv_2011
JOIN inv_2012 ON inv_2011.country_code = inv_2012.country_code
JOIN inv_2013 ON inv_2012.country_code = inv_2013.country_code
ORDER BY avg_inv_2011 DESC
;